In [2]:
%pip install statsmodels

  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.6 MB 1.5 MB/s eta 0:00:06
   ------ --------------------------------- 1.6/9.6 MB 2.9 MB/s eta 0:00:03
   ---------- ----------------------------- 2.6/9.6 MB 3.6 MB/s eta 0:00:02
   ------------- -------------------------- 3.1/9.6 MB 3.8 MB/s eta 0:00:02
   ------------------- -------------------- 4.7/9.6 MB 4.1 MB/s eta 0:00:02
   ------------------------ --------------- 5.8/9.6 MB 4.2 MB/s eta 0:00:01
   --------------------------- ------------ 6.6/9.6 MB 4.2 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.6 MB 4.2 MB/s eta 0:00:01
   --------------------------------- ------ 8.1/9.

In [10]:
import pandas as pd
import numpy as np
import os
import statsmodels.api as sm

# ==========================================
# 1. CARREGAMENTO E HIGIENIZAÇÃO DOS DADOS
# ==========================================
print("A carregar os ficheiros...")

pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"
#precos = pd.read_csv(os.path.join(pasta_dados, "precos_sanitizada_v3.csv"), index_col='Data', parse_dates=True)
precos = pd.read_csv(os.path.join(pasta_dados, "precos_sanitizada_v3.csv"))

#cdi = pd.read_excel('CDI_2010_2026.xlsx')
cdi = pd.read_excel(os.path.join(pasta_dados, "CDI_2010_2026.xlsx"))

#ibov = pd.read_excel('IBOV_2010_2026.xlsx')
ibov = pd.read_excel(os.path.join(pasta_dados, "IBOV_2010_2026.xlsx"))


# Uniformizar as colunas de data para o formato datetime do Pandas
precos['Data'] = pd.to_datetime(precos['Data'])
cdi['Data'] = pd.to_datetime(cdi['Data'])
ibov['Data'] = pd.to_datetime(ibov['Data'])

# Eliminar as colunas duplicadas antigas para limpar a memória
cdi = cdi.drop(columns=['Data'])
ibov = ibov.drop(columns=['Data'])

# ==========================================
# 2. CÁLCULO DOS RETORNOS SIMPLES DIÁRIOS
# ==========================================
print("A calcular retornos diários...")
retornos_ativos = precos.set_index('Data').pct_change().dropna(how='all')
retornos_ibov = ibov.set_index('Data').pct_change().dropna()
retornos_ibov.columns = ['IBOV']  # Nomear a coluna do mercado

# ==========================================
# 3. ALINHAMENTO TEMPORAL PERFEITO (MERGE)
# ==========================================
# Juntar ativos e Ibovespa pelas datas correspondentes
df_merged = retornos_ativos.merge(retornos_ibov, left_index=True, right_index=True)
# Juntar com o CDI (que já é a taxa diária, mantida conforme o Passo 1)
df_merged = df_merged.merge(cdi.set_index('Data'), left_index=True, right_index=True)
df_merged.rename(columns={'valor': 'CDI'}, inplace=True)

# ==========================================
# 4. EXECUÇÃO DO MODELO ECONOMÉTRICO CAPM
# ==========================================
print("A processar a regressão CAPM e rácios de risco...")
lista_resultados = []

# Filtramos apenas as colunas das ações (tudo exceto IBOV e CDI)
ativos = [col for col in retornos_ativos.columns]

# Parâmetros macroeconómicos anualizados da tua base para o Lucro Esperado
ibov_anual_ret = df_merged['IBOV'].mean() * 252
cdi_anual_ret = df_merged['CDI'].mean() * 252

for ativo in ativos:
    # Garantir que a coluna não está totalmente vazia
    if df_merged[ativo].dropna().empty:
        continue
        
    # Cálculo dos prémios de risco (Retorno em Excesso)
    y = df_merged[ativo] - df_merged['CDI']  # Variável Dependente
    x = df_merged['IBOV'] - df_merged['CDI'] # Variável Independente
    
    # Alinhar e remover eventuais NaNs temporários daquela ação específica
    dados_reg = pd.concat([y, x], axis=1).dropna()
    y_valid = dados_reg.iloc[:, 0]
    x_valid = dados_reg.iloc[:, 1]
    
    # Se houver dados suficientes, roda a Regressão Linear OLS
    if len(dados_reg) > 30:
        X = sm.add_constant(x_valid) # Adiciona o intercepto (Alfa)
        modelo = sm.OLS(y_valid, X).fit()
        
        alfa_diario = modelo.params.iloc[0]
        beta = modelo.params.iloc[1]
        
        # Anualização dos parâmetros para o relatório do TCC
        alfa_anual = alfa_diario * 252
        
        # Lucro Esperado Teórico via equação do CAPM
        retorno_esperado_capm = cdi_anual_ret + beta * (ibov_anual_ret - cdi_anual_ret)
        
        # --- Cálculo dos Rácios de Performance ---
        # Índice de Sharpe Anualizado
        ex_ret_mean = y_valid.mean()
        ex_ret_std = y_valid.std()
        sharpe = (ex_ret_mean / ex_ret_std) * np.sqrt(252) if ex_ret_std != 0 else np.nan
        
        # Índice de Sortino Anualizado (Volatilidade apenas dos retornos negativos)
        downside_std = y_valid[y_valid < 0].std()
        sortino = (ex_ret_mean / downside_std) * np.sqrt(252) if downside_std != 0 else np.nan
        
        # Guardar métricas no dicionário
        lista_resultados.append({
            'Ativo': ativo,
            'Beta (β)': round(beta, 3),
            'Alfa Anual (α)': f"{round(alfa_anual * 100, 2)}%",
            'Retorno Esp. CAPM': f"{round(retorno_esperado_capm * 100, 2)}%",
            'Índice Sharpe': round(sharpe, 2),
            'Índice Sortino': round(sortino, 2)
        })

# ==========================================
# 5. CONSOLIDAR COMPILAÇÃO ACADÉMICA
# ==========================================
df_capm_final = pd.DataFrame(lista_resultados)

# Exporta diretamente para CSV para copiares para a tua dissertação
df_capm_final.to_csv('Métricas_CAPM_e_Risco_TCC.csv', index=False)
print("\nSucesso! O ficheiro 'Métricas_CAPM_e_Risco_TCC.csv' foi gerado e está pronto para o teu Word!")
print(df_capm_final) # Mostra as primeiras 10 linhas no terminal

A carregar os ficheiros...
A calcular retornos diários...


KeyError: "None of ['Data'] are in the columns"